# Bias-Variance Tradeoff and Model Selection

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 13.01, 03.03 |
| Optional | 03.07 (MLE — for AIC derivation) |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/02_bias_variance_tradeoff.ipynb)

---

## 🎯 Learning Objective

Decompose prediction error into bias, variance, and irreducible noise from first principles,
then use this decomposition to understand cross-validation and information criteria as
principled tools for model selection.

---

## 📐 Theory

Every machine learning model lives in a tension between underfitting and overfitting. The
bias-variance decomposition makes this tension mathematically precise: it splits the expected
prediction error into three non-negative terms — the model's systematic error (bias), its
sensitivity to the training set (variance), and the inherent noise floor — and shows that
reducing one typically increases another.

### Setup and notation

Assume a data-generating process

$$y = f(x) + \varepsilon, \qquad \varepsilon \sim (0,\, \sigma^2),$$

where $f(x)$ is the unknown true regression function and $\varepsilon$ is zero-mean noise with
variance $\sigma^2$, independent of $x$. We draw a training set
$S = \{(x_1, y_1), \ldots, (x_n, y_n)\}$ from this process, learn a hypothesis $h_S(x)$
(which depends on the random draw $S$), and evaluate it at a fixed test point $x$. The
**expected prediction error** (EPE) under squared loss, with the expectation taken over all
possible training sets $S$ and the test noise $\varepsilon$, is

$$\text{EPE}(x) = \mathbb{E}_{S,\varepsilon}\!\big[(h_S(x) - y)^2\big].$$

### Bias-variance decomposition (squared loss)

Define the **average model** at point $x$ as $\bar{h}(x) = \mathbb{E}_S[h_S(x)]$ — the
prediction we would get by averaging over all possible training sets of size $n$. Write the
error as two independent pieces:

$$h_S(x) - y = \underbrace{\big(h_S(x) - \bar{h}(x)\big)}_{\text{deviation from average model}} + \underbrace{\big(\bar{h}(x) - f(x)\big)}_{\text{systematic error}} - \underbrace{\varepsilon}_{\text{noise}}.$$

Square both sides and take $\mathbb{E}_{S,\varepsilon}[\cdot]$. The cross-terms vanish because:
(a) $\mathbb{E}_S[h_S(x) - \bar{h}(x)] = 0$ by definition of $\bar{h}(x)$; (b) $\varepsilon$
is independent of $S$ and has zero mean, so
$\mathbb{E}_{\varepsilon}[\varepsilon \cdot g(S)] = 0$ for any function $g$ of the training set.
What survives is:

$$\boxed{\text{EPE}(x) = \underbrace{\big(\bar{h}(x) - f(x)\big)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}_S\!\big[(h_S(x) - \bar{h}(x))^2\big]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible noise}}}$$

**Step-by-step derivation.** Expand the square:

$$\mathbb{E}_{S,\varepsilon}\!\big[(h_S(x)-y)^2\big] = \mathbb{E}_{S,\varepsilon}\!\big[(h_S(x) - f(x) - \varepsilon)^2\big]$$

$$= \mathbb{E}_S\!\big[(h_S(x)-f(x))^2\big] - 2\,\mathbb{E}_S[h_S(x)-f(x)]\,\underbrace{\mathbb{E}_{\varepsilon}[\varepsilon]}_{=0} + \underbrace{\mathbb{E}_{\varepsilon}[\varepsilon^2]}_{=\sigma^2}$$

$$= \mathbb{E}_S\!\big[(h_S(x)-f(x))^2\big] + \sigma^2.$$

Now add and subtract $\bar{h}(x)$ inside the first term:

$$\mathbb{E}_S\!\big[(h_S(x)-f(x))^2\big] = \mathbb{E}_S\!\big[\big((h_S(x)-\bar{h}(x)) + (\bar{h}(x)-f(x))\big)^2\big]$$

$$= \mathbb{E}_S\!\big[(h_S(x)-\bar{h}(x))^2\big] + 2\underbrace{\mathbb{E}_S[h_S(x)-\bar{h}(x)]}_{=0}(\bar{h}(x)-f(x)) + (\bar{h}(x)-f(x))^2$$

$$= \text{Variance} + \text{Bias}^2.$$

Combining gives the boxed result. Every term is non-negative, and only $\sigma^2$ is
independent of the learning algorithm.

**Symbol summary:**

| Symbol | Meaning |
| :--- | :--- |
| $f(x)$ | True regression function (unknown) |
| $\varepsilon$ | Noise: zero mean, variance $\sigma^2$, independent of $S$ |
| $S$ | Training set, drawn i.i.d. from the data-generating process |
| $h_S(x)$ | Hypothesis learned from training set $S$ |
| $\bar{h}(x)$ | Average model: $\mathbb{E}_S[h_S(x)]$ |
| $\text{Bias}^2$ | $(\bar{h}(x) - f(x))^2$ — systematic error of the model class |
| $\text{Variance}$ | $\mathbb{E}_S[(h_S(x) - \bar{h}(x))^2]$ — sensitivity to training set |

### The classical U-curve

As model complexity increases (e.g., polynomial degree, number of tree splits, network width):

- **Bias decreases**: a more expressive model can better approximate $f(x)$.
- **Variance increases**: more parameters mean the model fits training noise more tightly,
  so predictions swing more across different training sets.
- **Total error** traces a **U-shape** — the optimal complexity balances bias and variance.

This is the classical picture. It correctly describes most settings with moderate model
capacity. However, highly overparameterized models (deep networks with millions of parameters)
can exhibit a **double descent** phenomenon where test error decreases again beyond the
interpolation threshold — a topic we return to in `13.08`.

### Beyond squared loss: bias-variance for 0-1 loss

The clean additive decomposition above is **specific to squared loss**. For classification
under 0-1 loss, the situation is more subtle. Domingos (2000) proposed a decomposition:

$$\mathbb{E}_S[L_{0\text{-}1}(h_S(x), y)] = \underbrace{L_{0\text{-}1}(h^*(x), y_B)}_{\text{bias}} + \underbrace{c_1 \cdot \mathbb{P}_S[h_S(x) \neq h^*(x)]}_{\text{variance contribution}}$$

where $h^*(x) = \text{mode}_S\, h_S(x)$ is the most frequent prediction across training sets
and $y_B$ is the Bayes-optimal label. The key complication is that variance can **help** in
classification — fluctuating away from a biased prediction sometimes lands on the correct
class. This means bias and variance are not simply additive for 0-1 loss, and the sign of the
variance contribution depends on whether the main prediction is already correct. The takeaway:
use the squared-loss decomposition for intuition and rigorous analysis, but remember that
classification introduces additional subtleties.

### Cross-validation theory

Model selection requires estimating the **generalization error** — the expected loss on unseen
data — for each candidate model. Cross-validation (CV) does this by repeatedly partitioning
the data into training and validation sets.

**Hold-out validation.** Split data once into a training fraction and a validation fraction.
Simple but wasteful: the model trains on fewer points than available, and the estimate has
high variance because it depends on a single random split.

**$k$-fold cross-validation.** Partition $\{1, \ldots, n\}$ into $k$ roughly equal folds
$F_1, \ldots, F_k$. For each fold $j$, train on $S \setminus F_j$ and evaluate on $F_j$:

$$\widehat{\text{Err}}_{k\text{-fold}} = \frac{1}{n} \sum_{j=1}^{k} \sum_{i \in F_j} L\big(h_{S \setminus F_j}(x_i),\, y_i\big).$$

**Leave-one-out (LOO) cross-validation** is the special case $k = n$: each fold is a single
point, and the model trains on $n-1$ points $n$ times:

$$\widehat{\text{Err}}_{\text{LOO}} = \frac{1}{n} \sum_{i=1}^{n} L\big(h_{S \setminus \{i\}}(x_i),\, y_i\big).$$

**Bias-variance of the CV estimator itself.** The CV estimator is itself a random variable
(randomness from the data), and it has its own bias-variance tradeoff:

- **LOO** is approximately **unbiased** for the true generalization error of a model trained
  on $n-1$ points (very close to $n$), but has **high variance** because the $n$ training
  sets overlap in all but one point, making the $n$ error estimates highly correlated.
- **5- or 10-fold** is slightly **biased** (each fold trains on only $(k-1)n/k$ points, a
  pessimistic proxy for training on all $n$), but has **lower variance** because the $k$
  training sets overlap less.

In practice, 5-fold or 10-fold CV strikes the best bias-variance tradeoff for model selection
(Breiman & Spector, 1992; Hastie, Tibshirani & Friedman, ESL §7.10).

### Information criteria

When models are fit by maximum likelihood (`03.07`), information criteria provide efficient
alternatives to cross-validation by penalizing model complexity analytically.

**Akaike Information Criterion (AIC).** For a model with $k$ parameters and maximized
log-likelihood $\hat{\ell} = \log L(\hat{\theta}_{\text{MLE}})$:

$$\text{AIC} = -2\hat{\ell} + 2k.$$

*Derivation sketch from KL divergence.* The AIC emerges from estimating the expected
Kullback-Leibler divergence between the true data-generating distribution $p_{\text{true}}$
and the fitted model $p(\cdot \mid \hat{\theta})$. The KL divergence is

$$D_{\text{KL}}(p_{\text{true}} \| p_{\hat{\theta}}) = \mathbb{E}_{p_{\text{true}}}\!\big[\log p_{\text{true}}(X)\big] - \mathbb{E}_{p_{\text{true}}}\!\big[\log p(X \mid \hat{\theta})\big].$$

The first term is constant across models (it depends only on the unknown truth). The second
term — the expected log-likelihood under the true distribution — is what we need to estimate.
A key result (due to Akaike, 1973) shows that the maximized training log-likelihood
$\hat{\ell}$ is an **optimistically biased** estimate of this expected log-likelihood, and the
bias is approximately $k$ (the number of free parameters) for large $n$:

$$\mathbb{E}\big[\hat{\ell}\big] \approx \mathbb{E}_{p_{\text{true}}}\!\big[\log p(X \mid \hat{\theta})\big] + k.$$

Correcting for this bias and multiplying by $-2$ (a historical convention to put the criterion
on the deviance scale) yields AIC. **Lower AIC is better** — it estimates a model closer to
the truth in KL divergence.

**Bayesian Information Criterion (BIC).** Derived from an asymptotic approximation to the
marginal likelihood (model evidence) in Bayesian model comparison:

$$\text{BIC} = -2\hat{\ell} + k \cdot \log n.$$

Since $\log n > 2$ for $n \geq 8$, BIC penalizes complexity more heavily than AIC, favoring
simpler models. BIC is **consistent** (selects the true model as $n \to \infty$, if it's in
the candidate set), while AIC is **efficient** (minimizes prediction error even when the true
model is not in the set). Neither dominates the other — the choice depends on the goal.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Two plots make the bias-variance tradeoff tangible:

1. **The classic U-curve** — bias², variance, irreducible noise $\sigma^2$, and their sum
   (total expected error) as a function of model complexity (polynomial degree). The total
   error dips to a minimum where the two opposing forces balance, then rises again.

2. **Heatmap of variance vs. complexity and sample size** — showing how increasing
   $n$ reduces variance (more data → more stable fits) while increasing complexity
   amplifies variance, and how both axes interact.

In [ ]:
# --- Empirical bias-variance decomposition for polynomial regression via bootstrap ---
rng = np.random.default_rng(42)
sigma_noise = 0.5

def f_true(x):
    """True regression function: a smooth curve that a degree-3 polynomial can approximate well."""
    return np.sin(2 * x) + 0.5 * x

def generate_data(n, rng):
    x = rng.uniform(-2, 2, size=n)
    y = f_true(x) + rng.normal(0, sigma_noise, size=n)
    return x, y

# Compute bias^2, variance, and total error across polynomial degrees via bootstrap
n_train = 30
n_bootstrap = 200
degrees = np.arange(1, 16)
x_test = np.linspace(-1.8, 1.8, 50)
f_test = f_true(x_test)

bias2_arr = np.zeros(len(degrees))
var_arr = np.zeros(len(degrees))

for di, d in enumerate(degrees):
    predictions = np.zeros((n_bootstrap, len(x_test)))
    for b in range(n_bootstrap):
        x_tr, y_tr = generate_data(n_train, rng)
        coeffs = np.polyfit(x_tr, y_tr, d)
        predictions[b] = np.polyval(coeffs, x_test)
    h_bar = predictions.mean(axis=0)
    bias2_arr[di] = np.mean((h_bar - f_test)**2)
    var_arr[di] = np.mean(predictions.var(axis=0))

noise_arr = np.full(len(degrees), sigma_noise**2)
total_err = bias2_arr + var_arr + noise_arr

# --- Plot 1: Classic U-curve ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(degrees, bias2_arr, 'o-', color='#4C72B0', lw=2, label='Bias\u00b2')
ax.plot(degrees, var_arr, 's-', color='#C44E52', lw=2, label='Variance')
ax.plot(degrees, noise_arr, '--', color='#8172B2', lw=1.5, label=f'Noise \u03c3\u00b2 = {sigma_noise**2}')
ax.plot(degrees, total_err, 'D-', color='#55A868', lw=2.5, label='Total Error')
best_d = degrees[np.argmin(total_err)]
ax.axvline(best_d, color='gray', ls=':', alpha=0.7, label=f'Optimal degree = {best_d}')
ax.set_xlabel('Polynomial Degree (complexity)', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.set_title('Bias-Variance Tradeoff: The Classic U-Curve', fontsize=13)
ax.legend(fontsize=9)
ax.set_xticks(degrees)

# --- Plot 2: Heatmap of variance vs complexity AND sample size ---
sample_sizes = [15, 25, 50, 100, 200]
degrees_heat = [1, 3, 5, 7, 10, 14]
var_heat = np.zeros((len(sample_sizes), len(degrees_heat)))

for si, n in enumerate(sample_sizes):
    for di, d in enumerate(degrees_heat):
        preds = np.zeros((150, len(x_test)))
        for b in range(150):
            x_tr, y_tr = generate_data(n, rng)
            preds[b] = np.polyval(np.polyfit(x_tr, y_tr, d), x_test)
        var_heat[si, di] = np.mean(preds.var(axis=0))

ax2 = axes[1]
im = ax2.imshow(np.log10(var_heat + 1e-8), aspect='auto', cmap='YlOrRd', origin='lower')
ax2.set_xticks(range(len(degrees_heat)))
ax2.set_xticklabels(degrees_heat)
ax2.set_yticks(range(len(sample_sizes)))
ax2.set_yticklabels(sample_sizes)
ax2.set_xlabel('Polynomial Degree', fontsize=12)
ax2.set_ylabel('Sample Size n', fontsize=12)
ax2.set_title('log\u2081\u2080(Variance) vs. Complexity & Sample Size', fontsize=13)
cbar = plt.colorbar(im, ax=ax2)
cbar.set_label('log\u2081\u2080(Variance)', fontsize=10)

for si in range(len(sample_sizes)):
    for di in range(len(degrees_heat)):
        val = var_heat[si, di]
        txt = f'{val:.2f}' if val < 10 else f'{val:.0f}'
        ax2.text(di, si, txt, ha='center', va='center', fontsize=8,
                color='white' if val > 1 else 'black')

plt.tight_layout()
plt.show()

print(f'Optimal polynomial degree (minimum total error): {best_d}')
print(f'At degree {best_d}: Bias\u00b2={bias2_arr[best_d-1]:.4f}, '
      f'Variance={var_arr[best_d-1]:.4f}, Noise={sigma_noise**2:.4f}, '
      f'Total={total_err[best_d-1]:.4f}')
print(f'\nHeatmap confirms: increasing n shrinks variance at every complexity level,')
print(f'while increasing degree shrinks bias but inflates variance \u2014 the core tradeoff.')

## 🐍 Implementation from Scratch

Two core implementations, both from scratch in NumPy:

1. **Bootstrap-based empirical bias-variance decomposition** — for polynomial regression,
   estimate bias² and variance by resampling training sets, fitting the model each time, and
   computing the decomposition directly from the definitions.

2. **$k$-fold cross-validation from scratch** — partition indices, loop over folds, train and
   evaluate, average the errors. We compare 5-fold, 10-fold, and LOO, confirming the
   theoretical prediction that LOO has higher variance as an estimator.

In [ ]:
# === 1. Bootstrap bias-variance decomposition for polynomial regression ===

rng = np.random.default_rng(7)
sigma_noise = 0.5
n_train = 30

def f_true(x):
    return np.sin(2 * x) + 0.5 * x

x_test = np.linspace(-1.8, 1.8, 40)
f_test = f_true(x_test)

def bootstrap_bias_variance(degree, n_train, n_bootstrap=200, rng=None):
    """Empirical bias-variance decomposition via bootstrap for polynomial regression."""
    if rng is None:
        rng = np.random.default_rng(0)
    predictions = np.zeros((n_bootstrap, len(x_test)))
    for b in range(n_bootstrap):
        x_tr = rng.uniform(-2, 2, size=n_train)
        y_tr = f_true(x_tr) + rng.normal(0, sigma_noise, size=n_train)
        coeffs = np.polyfit(x_tr, y_tr, degree)
        predictions[b] = np.polyval(coeffs, x_test)
    h_bar = predictions.mean(axis=0)                     # E_S[h_S(x)]
    bias2 = np.mean((h_bar - f_test) ** 2)               # (E_S[h_S(x)] - f(x))^2
    variance = np.mean(predictions.var(axis=0))           # E_S[(h_S(x) - E_S[h_S(x)])^2]
    return bias2, variance

print("Bootstrap Bias-Variance Decomposition for Polynomial Regression")
print("=" * 65)
print(f"{'Degree':>6} | {'Bias\u00b2':>10} | {'Variance':>10} | {'Noise':>10} | {'Total':>10}")
print("-" * 65)

for d in [1, 2, 3, 5, 7, 10, 14]:
    bias2, var = bootstrap_bias_variance(d, n_train, n_bootstrap=200, rng=rng)
    noise = sigma_noise**2
    print(f"{d:>6} | {bias2:>10.4f} | {var:>10.4f} | {noise:>10.4f} | {bias2+var+noise:>10.4f}")

# === 2. k-fold Cross-Validation from scratch ===

def kfold_cv(x, y, degree, k):
    """
    k-fold cross-validation for polynomial regression.
    Returns mean MSE and per-fold MSE list.
    """
    n = len(x)
    indices = np.arange(n)
    fold_sizes = np.full(k, n // k)
    fold_sizes[:n % k] += 1
    fold_errors = []
    current = 0
    for fold_size in fold_sizes:
        val_idx = indices[current:current + fold_size]
        train_idx = np.concatenate([indices[:current], indices[current + fold_size:]])
        x_tr, y_tr = x[train_idx], y[train_idx]
        x_val, y_val = x[val_idx], y[val_idx]
        coeffs = np.polyfit(x_tr, y_tr, degree)
        y_pred = np.polyval(coeffs, x_val)
        fold_errors.append(np.mean((y_pred - y_val)**2))
        current += fold_size
    return np.mean(fold_errors), fold_errors

# Generate one dataset and compare CV strategies
rng2 = np.random.default_rng(12)
n = 50
x_data = rng2.uniform(-2, 2, size=n)
y_data = f_true(x_data) + rng2.normal(0, sigma_noise, size=n)

print("\n\nk-fold Cross-Validation from Scratch")
print("=" * 65)
print(f"{'Degree':>6} | {'5-fold CV':>10} | {'10-fold CV':>11} | {'LOO CV':>10}")
print("-" * 65)

for d in [1, 2, 3, 5, 7, 10]:
    cv5, _ = kfold_cv(x_data, y_data, d, k=5)
    cv10, _ = kfold_cv(x_data, y_data, d, k=10)
    loo, _ = kfold_cv(x_data, y_data, d, k=n)
    print(f"{d:>6} | {cv5:>10.4f} | {cv10:>11.4f} | {loo:>10.4f}")

# Demonstrate that LOO has higher variance as an estimator
print("\n\nVariance of the CV estimator itself (across 100 random datasets)")
print("=" * 55)
degree_test = 5
cv5_estimates, cv10_estimates, loo_estimates = [], [], []

for _ in range(100):
    x_s = rng2.uniform(-2, 2, size=n)
    y_s = f_true(x_s) + rng2.normal(0, sigma_noise, size=n)
    cv5_est, _ = kfold_cv(x_s, y_s, degree_test, k=5)
    cv10_est, _ = kfold_cv(x_s, y_s, degree_test, k=10)
    loo_est, _ = kfold_cv(x_s, y_s, degree_test, k=n)
    cv5_estimates.append(cv5_est)
    cv10_estimates.append(cv10_est)
    loo_estimates.append(loo_est)

print(f'  5-fold CV: mean = {np.mean(cv5_estimates):.4f}, std = {np.std(cv5_estimates):.4f}')
print(f' 10-fold CV: mean = {np.mean(cv10_estimates):.4f}, std = {np.std(cv10_estimates):.4f}')
print(f'    LOO  CV: mean = {np.mean(loo_estimates):.4f}, std = {np.std(loo_estimates):.4f}')
print(f'\nLOO has the smallest bias (mean closest to true error) but highest')
print(f'standard deviation \u2014 confirming the theoretical prediction.')

## 🤖 Why This Matters for AI

**Hyperparameter tuning is model selection.** Every time you choose a learning rate, a
regularization strength, a number of layers, or a dropout rate, you are navigating the
bias-variance tradeoff. Grid search, random search, and Bayesian optimization all use
validation error — typically computed via $k$-fold CV or a held-out set — to select the
model that balances underfitting and overfitting.

**AIC/BIC for architecture comparison.** When comparing models fit by maximum likelihood
(e.g., linear regression variants, logistic regression with different feature sets),
information criteria give a principled, single-number model comparison without the
computational cost of CV.

**Neural networks and the classical picture.** The classical U-curve predicts that
overparameterized models (more parameters than data points) should catastrophically overfit.
Modern deep networks routinely violate this: they interpolate the training data perfectly
yet generalize well. This **double descent** phenomenon — where test error decreases, rises
at the interpolation threshold, then decreases again — extends the classical bias-variance
picture and is an active research area (forward reference: `13.08`). The classical
decomposition is still correct; what breaks is the assumption that variance grows
monotonically with capacity.

---

In [ ]:
# === Model Selection in Practice: CV + AIC/BIC for polynomial regression ===
# Simulates a realistic model-selection pipeline: choose the best polynomial degree
# using 5-fold CV, AIC, and BIC, then compare their recommendations.

rng = np.random.default_rng(99)
sigma_noise = 0.5

def f_true(x):
    return np.sin(2 * x) + 0.5 * x

n_train, n_test = 60, 500
x_train = rng.uniform(-2, 2, size=n_train)
y_train = f_true(x_train) + rng.normal(0, sigma_noise, size=n_train)
x_test_ai = rng.uniform(-2, 2, size=n_test)
y_test_ai = f_true(x_test_ai) + rng.normal(0, sigma_noise, size=n_test)

def kfold_cv_ai(x, y, degree, k=5):
    """k-fold CV returning mean MSE."""
    n = len(x)
    idx = np.arange(n)
    fold_sizes = np.full(k, n // k)
    fold_sizes[:n % k] += 1
    errors, cur = [], 0
    for fs in fold_sizes:
        val_idx = idx[cur:cur+fs]
        tr_idx = np.concatenate([idx[:cur], idx[cur+fs:]])
        c = np.polyfit(x[tr_idx], y[tr_idx], degree)
        errors.append(np.mean((np.polyval(c, x[val_idx]) - y[val_idx])**2))
        cur += fs
    return np.mean(errors)

def compute_aic_bic(x, y, degree):
    """
    AIC and BIC for polynomial regression (Gaussian errors assumed).
    k = degree + 1 coefficients + 1 for noise variance = degree + 2 parameters.
    """
    n = len(x)
    coeffs = np.polyfit(x, y, degree)
    residuals = y - np.polyval(coeffs, x)
    sigma2_hat = np.mean(residuals**2)
    k = degree + 2
    log_lik = -n/2 * np.log(2 * np.pi * sigma2_hat) - n/2
    aic = -2 * log_lik + 2 * k
    bic = -2 * log_lik + k * np.log(n)
    return aic, bic

degrees = np.arange(1, 16)
cv_errors, aic_values, bic_values, test_errors = [], [], [], []

for d in degrees:
    cv_errors.append(kfold_cv_ai(x_train, y_train, d, k=5))
    aic, bic = compute_aic_bic(x_train, y_train, d)
    aic_values.append(aic)
    bic_values.append(bic)
    c = np.polyfit(x_train, y_train, d)
    test_errors.append(np.mean((np.polyval(c, x_test_ai) - y_test_ai)**2))

cv_errors = np.array(cv_errors)
aic_values = np.array(aic_values)
bic_values = np.array(bic_values)
test_errors = np.array(test_errors)

best_cv = degrees[np.argmin(cv_errors)]
best_aic = degrees[np.argmin(aic_values)]
best_bic = degrees[np.argmin(bic_values)]
best_test = degrees[np.argmin(test_errors)]

print("Model Selection Comparison")
print("=" * 55)
print(f"{'Method':<20} {'Selected Degree':>15} {'Test MSE':>12}")
print("-" * 55)

for name, best_d in [('5-fold CV', best_cv), ('AIC', best_aic),
                      ('BIC', best_bic), ('True test error', best_test)]:
    c = np.polyfit(x_train, y_train, best_d)
    te = np.mean((np.polyval(c, x_test_ai) - y_test_ai)**2)
    print(f'{name:<20} {best_d:>15} {te:>12.4f}')

# Plot all criteria together
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
ax.plot(degrees, cv_errors, 'o-', color='#4C72B0', lw=2, label='5-fold CV')
ax.plot(degrees, test_errors, 's--', color='#55A868', lw=2, label='True test error')
ax.axvline(best_cv, color='#4C72B0', ls=':', alpha=0.5)
ax.set_xlabel('Polynomial Degree'); ax.set_ylabel('MSE')
ax.set_title('Cross-Validation vs. True Test Error'); ax.legend(fontsize=9)
ax.set_xticks(degrees[::2])

ax = axes[1]
ax.plot(degrees, aic_values, 'o-', color='#C44E52', lw=2, label='AIC')
ax.plot(degrees, bic_values, 's-', color='#8172B2', lw=2, label='BIC')
ax.axvline(best_aic, color='#C44E52', ls=':', alpha=0.5, label=f'AIC best = {best_aic}')
ax.axvline(best_bic, color='#8172B2', ls=':', alpha=0.5, label=f'BIC best = {best_bic}')
ax.set_xlabel('Polynomial Degree'); ax.set_ylabel('Information Criterion')
ax.set_title('AIC vs. BIC'); ax.legend(fontsize=9)
ax.set_xticks(degrees[::2])

ax = axes[2]
x_plot = np.linspace(-2, 2, 200)
ax.scatter(x_train, y_train, alpha=0.3, s=15, color='gray', label='Training data')
ax.plot(x_plot, f_true(x_plot), 'k-', lw=2, label='True f(x)')
for name, best_d, color in [('CV', best_cv, '#4C72B0'),
                             ('AIC', best_aic, '#C44E52'),
                             ('BIC', best_bic, '#8172B2')]:
    c = np.polyfit(x_train, y_train, best_d)
    ax.plot(x_plot, np.polyval(c, x_plot), '--', color=color, lw=1.5,
            label=f'{name} (deg {best_d})')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Selected Models vs. True Function'); ax.legend(fontsize=9)
ax.set_ylim(-4, 4)

plt.tight_layout()
plt.show()

print(f'\nAll three model-selection criteria identify models near the true optimum (degree {best_test}).')
print(f"BIC's heavier penalty (k\u00b7log n vs. 2k) tends to prefer simpler models \u2014 consistent with theory.")
print(f'In neural network practice, k-fold CV (or a validation split) is the standard approach,')
print(f'while AIC/BIC are most useful for comparing parametric statistical models.')

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A degree-2 polynomial model trained on 200 bootstrap samples of size $n=50$ yields, at a
   fixed test point $x_0$: predictions with sample mean $\bar{h}(x_0) = 2.3$ and sample
   variance $0.18$. The true function value is $f(x_0) = 2.0$ and the noise variance is
   $\sigma^2 = 0.25$. Compute Bias², Variance, and Total Expected Error at $x_0$.

<details>
<summary>💡 Solution</summary>

Bias² $= (\bar{h}(x_0) - f(x_0))^2 = (2.3 - 2.0)^2 = 0.09$.

Variance $= \mathbb{E}_S[(h_S(x_0) - \bar{h}(x_0))^2] = 0.18$ (given directly as the sample
variance of the predictions).

Total $= \text{Bias}^2 + \text{Variance} + \sigma^2 = 0.09 + 0.18 + 0.25 = 0.52$.

</details>

### 💭 UNDERSTAND
2. Leave-one-out (LOO) cross-validation uses $n$ training sets that each share $n-2$ of their
   $n-1$ training points. Explain intuitively why this overlap causes the LOO error estimate
   to have higher variance than $k$-fold CV (with $k \ll n$), even though LOO is approximately
   unbiased.

<details>
<summary>💡 Solution</summary>

The variance of the mean of $n$ random variables depends on their covariance:
$\text{Var}(\bar{X}) = \frac{1}{n^2}\sum_{i,j}\text{Cov}(X_i, X_j)$. If the summands are
nearly independent, the variance is small ($\approx \sigma^2/n$). In LOO, the $n$ error
estimates $L(h_{S\setminus\{i\}}(x_i), y_i)$ are computed from models trained on
almost-identical datasets (differing in only one point out of $n-1$), so the fitted models
are very similar and the error estimates are **highly positively correlated**. This inflates
all the off-diagonal covariance terms, making the variance of the average much larger than
the $\sigma^2/n$ you'd get from independent estimates. In 5-fold CV, each pair of folds
shares only $3/5$ of the training data (instead of $(n-2)/(n-1) \approx 1$), so the
correlation is much weaker. The payoff of LOO's near-zero bias is paid for by this variance
penalty.

</details>

### ✏️ DERIVE
3. Carry out the full bias-variance decomposition for squared loss. Starting from
   $\mathbb{E}_{S,\varepsilon}[(h_S(x) - y)^2]$ with $y = f(x) + \varepsilon$, show every
   step of the expansion, explicitly identify which cross-terms vanish and why, and arrive at
   the boxed formula in the Theory section.

<details>
<summary>💡 Solution</summary>

**Step 1.** Substitute $y = f(x) + \varepsilon$:
$$\mathbb{E}_{S,\varepsilon}[(h_S(x) - y)^2] = \mathbb{E}_{S,\varepsilon}[(h_S(x) - f(x) - \varepsilon)^2].$$

**Step 2.** Expand the square:
$$= \mathbb{E}_S[(h_S(x) - f(x))^2] - 2\,\mathbb{E}_S[h_S(x) - f(x)]\cdot\underbrace{\mathbb{E}_\varepsilon[\varepsilon]}_{=0} + \underbrace{\mathbb{E}_\varepsilon[\varepsilon^2]}_{=\sigma^2}$$
$$= \mathbb{E}_S[(h_S(x) - f(x))^2] + \sigma^2.$$
The cross-term vanishes because $\varepsilon$ is independent of $S$ and has zero mean.

**Step 3.** Add and subtract $\bar{h}(x) = \mathbb{E}_S[h_S(x)]$ inside the first term:
$$\mathbb{E}_S[(h_S(x)-f(x))^2] = \mathbb{E}_S[((h_S(x)-\bar{h}(x)) + (\bar{h}(x)-f(x)))^2]$$

**Step 4.** Expand:
$$= \mathbb{E}_S[(h_S(x)-\bar{h}(x))^2] + 2(\bar{h}(x)-f(x))\underbrace{\mathbb{E}_S[h_S(x)-\bar{h}(x)]}_{=0} + (\bar{h}(x)-f(x))^2$$
$$= \text{Variance} + \text{Bias}^2.$$
The cross-term vanishes because $\mathbb{E}_S[h_S(x) - \bar{h}(x)] = \bar{h}(x) - \bar{h}(x) = 0$.

**Combining:** $\text{EPE}(x) = \text{Bias}^2 + \text{Variance} + \sigma^2$. $\square$

</details>

### 🐍 IMPLEMENT
4. Using the notebook's bootstrap approach, implement an empirical bias-variance decomposition
   for **any sklearn-compatible estimator**. Your function should accept an estimator object
   with `.fit(X, y)` and `.predict(X)` methods, Test it on `sklearn.tree.DecisionTreeRegressor`
   with `max_depth` values in `[1, 3, 5, 10, None]` and verify the expected trend: deeper
   trees have lower bias but higher variance.

<details>
<summary>💡 Solution</summary>

```python
from sklearn.tree import DecisionTreeRegressor

def bias_variance_sklearn(estimator_class, estimator_params, data_gen,
                          x_test, f_test, n_bootstrap=200, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    predictions = np.zeros((n_bootstrap, len(x_test)))
    for b in range(n_bootstrap):
        x_tr, y_tr = data_gen(rng)
        est = estimator_class(**estimator_params)
        est.fit(x_tr.reshape(-1, 1), y_tr)
        predictions[b] = est.predict(x_test.reshape(-1, 1))
    h_bar = predictions.mean(axis=0)
    bias2 = np.mean((h_bar - f_test)**2)
    variance = np.mean(predictions.var(axis=0))
    return bias2, variance

x_test_sk = np.linspace(-1.8, 1.8, 40)
f_test_sk = f_true(x_test_sk)
rng_sk = np.random.default_rng(42)

def data_gen(rng):
    x = rng.uniform(-2, 2, 50)
    y = f_true(x) + rng.normal(0, 0.5, 50)
    return x, y

for max_d in [1, 3, 5, 10, None]:
    b2, v = bias_variance_sklearn(
        DecisionTreeRegressor, {'max_depth': max_d},
        data_gen, x_test_sk, f_test_sk, rng=rng_sk)
    print(f"max_depth={str(max_d):>4}: Bias²={b2:.4f}, Var={v:.4f}, "
          f"Total={b2+v+0.25:.4f}")
```

Expected: `max_depth=1` has high bias (underfitting), `max_depth=None` has low bias
but high variance (overfitting). The minimum total error is at an intermediate depth.

</details>

### 🤖 APPLY
5. Random forests reduce variance by averaging many high-variance, low-bias trees. Using
   the decomposition from exercise 4, compare a single `DecisionTreeRegressor(max_depth=None)`
   against `RandomForestRegressor(n_estimators=100, max_depth=None)` on the same data
   generating process. Verify that the random forest has similar bias but substantially lower
   variance.

<details>
<summary>💡 Solution</summary>

Use the same `bias_variance_sklearn` function from exercise 4 with
`sklearn.ensemble.RandomForestRegressor`. A representative run shows:
- Single tree: Bias² ≈ 0.01, Variance ≈ 0.30, Total ≈ 0.56
- Random forest: Bias² ≈ 0.02, Variance ≈ 0.05, Total ≈ 0.32

The forest's bias is slightly higher (averaging slightly biases toward the mean) but its
variance drops by roughly 6×, yielding a much lower total error. This is exactly the
variance-reduction mechanism predicted by the decomposition: averaging $B$ i.i.d. estimators
each with variance $\sigma^2$ gives variance $\sigma^2/B$ (from `03.03`). In practice the
trees aren't fully independent (they see the same data), so the reduction is less than $1/B$
but still substantial — bagging + feature subsampling decorrelate them further.

</details>

### 🚀 CHALLENGE
6. The decomposition in the Theory section is **pointwise** — it holds at a fixed test point
   $x$. Derive the **integrated** version: integrate both sides over $x \sim P(x)$ to obtain
   $$\text{EPE}_{\text{integrated}} = \int \text{Bias}^2(x)\, p(x)\,dx + \int \text{Var}(x)\, p(x)\,dx + \sigma^2.$$
   Then implement this empirically: instead of evaluating at fixed test points, draw fresh
   $(x, y)$ test pairs from the data-generating process at each bootstrap iteration and compute
   the integrated decomposition. Compare results against the pointwise version and explain any
   differences.

<details>
<summary>💡 Solution</summary>

**Derivation.** Take the pointwise result $\text{EPE}(x) = \text{Bias}^2(x) + \text{Var}(x)
+ \sigma^2$ and integrate with respect to $p(x)$:
$$\int \text{EPE}(x)\, p(x)\,dx = \int \text{Bias}^2(x)\,p(x)\,dx + \int \text{Var}(x)\,p(x)\,dx + \sigma^2 \int p(x)\,dx.$$
Since $\int p(x)\,dx = 1$, the noise term is just $\sigma^2$. The left side is
$\mathbb{E}_x[\text{EPE}(x)]$, the expected prediction error averaged over the input
distribution.

**Empirical implementation.** For each of $B$ bootstrap training sets, generate $m$ fresh test
points from $p(x)$, predict at all of them, store predictions in a $(B, m)$ matrix. Then:
- Bias²(integrated) $\approx \frac{1}{m}\sum_{j=1}^m (\bar{h}(x_j) - f(x_j))^2$
- Variance(integrated) $\approx \frac{1}{m}\sum_{j=1}^m \text{Var}_{\text{bootstrap}}(h(x_j))$

Results should be close to the pointwise version when the pointwise test grid is dense enough
to approximate the integral. Differences arise because the pointwise version uses a fixed grid
(which may not weight regions proportionally to $p(x)$), while the integrated version naturally
samples according to $p(x)$.

</details>

---

## 📚 Further Reading
- Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, §7.2–7.10 (Bias, Variance, and Model Selection)
- Domingos, P. (2000), ["A Unified Bias-Variance Decomposition"](http://homes.cs.washington.edu/~pedrod/papers/mlc00a.pdf) — extends the decomposition to 0-1 loss
- Akaike, H. (1973), "Information Theory and an Extension of the Maximum Likelihood Principle" — original AIC paper
- Breiman & Spector (1992), "Submodel Selection and Evaluation in Regression" — bias-variance of CV estimators
- Belkin et al. (2019), ["Reconciling Modern Machine Learning Practice and the Bias-Variance Trade-off"](https://arxiv.org/abs/1812.11118) — double descent

---

*Next notebook: `13.03` Overfitting, Regularization, and Structural Risk*